# Object Detection with VGG16: Multi-Output Learning
This code demonstrates how to build a custom object detection model using a pre-trained **VGG16** backbone. 

### The Architecture:
1.  **Backbone**: VGG16 (Feature Extractor)
2.  **Regression Head**: Predicts the bounding box coordinates $[x_{min}, y_{min}, x_{max}, y_{max}]$.
3.  **Classification Head**: Predicts the object category (e.g., Dog, Cat, Car).

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Flatten, Dense, Input, Dropout
from tensorflow.keras.models import Model
import numpy as np

print(f"TensorFlow version: {tf.__version__}")

## 1. Load the Backbone (Feature Extractor)
We use VGG16 pre-trained on ImageNet. We set `include_top=False` to remove the fully connected layers at the end, allowing us to attach our own custom heads.

In [ ]:
# Define input shape
input_shape = (224, 224, 3)
input_tensor = Input(shape=input_shape)

# Load VGG16
base_model = VGG16(weights='imagenet', include_top=False, input_tensor=input_tensor)

# Freeze the backbone to preserve learned features during initial training
base_model.trainable = False

print("Backbone loaded and frozen.")

## 2. Building the Dual Heads
We flatten the output of the VGG16 base and split the network into two paths.

In [ ]:
# Flatten the features from the backbone
flatten = Flatten()(base_model.output)

### Path A: Bounding Box Regression ###
# Purpose: Predict 4 coordinates. Activation 'sigmoid' is used if coordinates are normalized [0, 1].
bbox_head = Dense(128, activation="relu")(flatten)
bbox_head = Dense(64, activation="relu")(bbox_head)
bbox_head = Dense(32, activation="relu")(bbox_head)
bbox_head = Dense(4, activation="sigmoid", name="bounding_box")(bbox_head)

### Path B: Classification ###
# Purpose: Predict the class of the object within the box.
class_head = Dense(512, activation="relu")(flatten)
class_head = Dropout(0.5)(class_head)
class_head = Dense(256, activation="relu")(class_head)
class_head = Dense(10, activation="softmax", name="class_label")(class_head)

## 3. Define and Compile the Model
We create a `Model` with one input and two distinct outputs. We then assign specific loss functions to each output name.

In [ ]:
# Construct the multi-output model
model = Model(inputs=base_model.input, outputs=[bbox_head, class_head])

# Compile with specialized losses
model.compile(
    optimizer='adam',
    loss={
        'bounding_box': 'mean_squared_error',   # For regression (coordinates)
        'class_label': 'sparse_categorical_crossentropy' # For integer labels
    },
    metrics={
        'bounding_box': 'mse',
        'class_label': 'accuracy'
    }
)

model.summary()

## 4. Preparing the Data (Format Guide)
To train this model, your labels must be structured as a dictionary corresponding to the output names defined in the model.

In [ ]:
"""
EXAMPLE DATA FORMAT:

# Assume X_train is your (N, 224, 224, 3) image array

train_targets = {
    "bounding_box": np.array([[0.1, 0.1, 0.5, 0.5], ...]), # Normalized coordinates
    "class_label": np.array([1, 5, 2, ...])               # Class indices
}

model.fit(X_train, train_targets, epochs=10, batch_size=32)
"""
print("Ready for data integration.")